# HADDOCK3 redocking exploration -- per protein, per macro-conformation

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

Same shape as `ga1_pocket_exploration.ipynb` (per-residue Rosetta REU
decomposition, same `resnr_label` pocket-position annotation), applied to
`redocking/`'s independent, physics-based HADDOCK3 redocking of GA1
instead of ABCfold's own ab initio poses -- see `redocking/README.md` and
`rescoring/README.md`'s "Scoring redocking/'s HADDOCK3-redocked poses"
section for the full pipeline this reads from.

**What's different from `ga1_pocket_exploration.ipynb`, and why:**
- **One scored pose per (protein, `ca_cluster`), not thousands of pooled
  frames.** `rescoring/src/rescore_redocked_batch.py` scores exactly one
  model (HADDOCK3's own top-ranked model) per redocked complex, with no
  FastRelax and no stochastic replicas -- so every bar plot below is that
  one pose's real per-residue energy, not a mean over an ensemble. A
  `pose_label` of `ca0`/`ca1`/`ca2` here means "this protein's macro-
  conformation cluster `ca_cluster`", not a further ligand-pose sub-
  cluster (redocking has no equivalent of ABCfold's own multi-frame ligand
  pose clustering -- HADDOCK3 already samples/ranks its own pose ensemble
  internally, `rigidbody sampling=200 -> seletop=40 -> flexref`, and only
  the single best-scoring result is kept).
- **Both importer AND non-importer proteins have a GA1 pose here.**
  HADDOCK3 docks GA1 into every manifest complex regardless of role
  (that's the whole point -- an independent physics-based check that
  doesn't already know a protein's importer/non-importer label) -- see
  the importer-vs-non-importer energetics/CDD-agreement sections below,
  which have no equivalent in `ga1_pocket_exploration.ipynb` (there is no
  "non-importer GA1 pose" anywhere else in this pipeline).
- **No PLIP/ChimeraX-minimized pass exists for these poses** -- stages
  14-16 were never run against `redocking/`'s output, only against
  ABCfold's own poses -- so the "Interaction type" and "Atom-level
  interaction matrices" sections have no equivalent here. What replaces
  them: the CDD-pocket precision/recall agreement and the sequence-LDA-
  importance-vs-Rosetta-energy overlay `rescore_redocked_aggregate.py`
  already computes, plus Stage 7's RMSD-vs-ABCfold-pose /
  pocket-contact-count comparison.

Standalone exploratory notebook, not part of the automated Snakemake
pipeline -- run after `redocking/`'s Stage 6 (HADDOCK3 array) and
`rescoring/src/rescore_redocked_{batch,aggregate}.py` /
`redocking/src/compare_to_abcfold.py` (Stage 7) have produced
`redocking/results/rescoring/{all_contacts,cdd_agreement,lda_unfavorable_contacts}.csv`
and `redocking/results/comparison/{summary.csv,*_comparison.json}`.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path("..")
REDOCKING_ROOT = ROOT / "redocking"
RESCORING_ROOT = REDOCKING_ROOT / "results" / "rescoring"
COMPARISON_ROOT = REDOCKING_ROOT / "results" / "comparison"

LIGAND = "GA1"

ROLE_COLORS = {"importer": "#2ca02c", "non_importer": "#7f7f7f"}


## Load data

`all_contacts.csv` is `rescore_redocked_aggregate.py`'s own output --
already pooled across every scored complex and already position-mapped
(`position` = CDD pocket position 1-35, NaN outside the pocket) via the
same `data/position_resnr_map.csv` `ga1_pocket_exploration.ipynb` uses, so
no manifest re-join is needed here the way that notebook needs one (that
notebook's `all_contacts.csv` can carry a *stale* embedded `ca_cluster` --
see its own markdown -- this project's `all_contacts.csv` doesn't have
that history: every per-complex CSV it pools is written fresh, in one
shot, straight from that complex's own manifest row).

`resnr_label` reuses the exact "raw resnr (pocket position)" / bare
"raw resnr" convention `ga1_pocket_exploration.ipynb` introduced -- built
directly from `all_contacts.csv`'s own `prot_resi`/`position` columns
rather than re-loading `position_resnr_map.csv` separately.


In [ ]:
all_contacts = pd.read_csv(RESCORING_ROOT / "all_contacts.csv")
all_contacts["pose_label"] = "ca" + all_contacts["ca_cluster"].astype(int).astype(str)


def _resnr_label(row) -> str:
    return f"{int(row.prot_resi)} ({int(row.position)})" if pd.notna(row.position) else str(int(row.prot_resi))


all_contacts["resnr_label"] = all_contacts.apply(_resnr_label, axis=1)

GA1_PROTEINS = sorted(all_contacts["protein"].unique())
n_importer_complexes = all_contacts.loc[all_contacts["role"] == "importer", "complex_id"].nunique()
n_non_importer_complexes = all_contacts.loc[all_contacts["role"] == "non_importer", "complex_id"].nunique()

print(f"{len(all_contacts)} contact rows, {all_contacts['complex_id'].nunique()} redocked complexes, "
      f"{len(GA1_PROTEINS)} proteins ({n_importer_complexes} importer, {n_non_importer_complexes} non_importer complexes)")
print(GA1_PROTEINS)


## Residue (AA) contribution

One bar plot per `(protein, ca_cluster)` -- x = raw protein residue number
(pocket position in parentheses where the residue falls in the CDD
pocket), y = that single redocked pose's own Rosetta REF2015
ligand<->residue two-body total (REU), bar color also encodes the value
(RdBu_r, centered at 0). Unlike `ga1_pocket_exploration.ipynb`'s version
of this plot, there is no averaging across complexes/replicas here -- each
bar is one real number from one real pose, since `rescore_redocked_batch.py`
scores exactly one model per redocked complex with no FastRelax.

Same clash-flagging convention as the ab-initio notebook: a cluster whose
worst residue reaches `ANOMALY_TWOBODY_REU` REU or more is flagged
directly in its title rather than silently plotted on the same scale as a
well-behaved cluster.


In [ ]:
ANOMALY_TWOBODY_REU = 10.0


def residue_contribution_table(protein: str) -> pd.DataFrame:
    """One row per (pose_label, prot_resi) for one protein -- this
    redocked complex's own Rosetta two-body total, no averaging (see
    markdown above)."""
    sub = all_contacts[all_contacts["protein"] == protein]
    per_edge = sub.drop_duplicates(["complex_id", "prot_resi"])
    return per_edge[["pose_label", "prot_resi", "resnr_label", "prot_resn", "twobody_total", "haddock_score"]].copy()


def plot_residue_contribution(protein: str):
    table = residue_contribution_table(protein)
    if table.empty:
        print(f"{protein}: no mapped Rosetta contacts")
        return
    for pose_label in sorted(table["pose_label"].unique()):
        sub = table[table["pose_label"] == pose_label].sort_values("prot_resi")
        worst = sub["twobody_total"].max()
        haddock_score = sub["haddock_score"].iloc[0]
        flag = (
            f"  \u26a0 ANOMALOUS -- max {worst:.1f} REU >= {ANOMALY_TWOBODY_REU:.0f} REU (likely clash)"
            if worst >= ANOMALY_TWOBODY_REU else ""
        )
        fig = px.bar(
            sub, x="resnr_label", y="twobody_total", color="twobody_total",
            color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
            labels=dict(twobody_total="two-body (REU)", resnr_label="protein residue number (pocket position)"),
            title=f"{protein} -- {pose_label} (HADDOCK score {haddock_score:.1f}){flag}",
        )
        fig.update_xaxes(title="protein residue number (pocket position)", type="category")
        fig.update_yaxes(title="two-body (REU)")
        fig.update_layout(coloraxis_showscale=False)
        fig.show()


## Unfavorable residue decomposition

For every residue with a net-*unfavorable* (positive REU) two-body total
above, what that positive total is actually made of, split by Rosetta's
own per-scoretype terms -- same decomposition and same interpretation as
`ga1_pocket_exploration.ipynb`'s version (`fa_rep`-dominated = steric
clash, `fa_elec`/`fa_sol`-dominated = electrostatic/desolvation mismatch).


In [ ]:
SCORETYPE_COLORS = {
    "fa_atr": "#2ca02c",
    "fa_rep": "#d62728",
    "fa_sol": "#ff7f0e",
    "fa_elec": "#1f77b4",
    "lk_ball_wtd": "#9467bd",
    "hbond_sc": "#17becf",
    "hbond_bb_sc": "#8c564b",
}
SCORETYPE_ORDER = list(SCORETYPE_COLORS)


def scoretype_decomposition_table(protein: str) -> pd.DataFrame:
    sub = all_contacts[all_contacts["protein"] == protein]
    per_edge = sub.drop_duplicates(["complex_id", "prot_resi", "scoretype"])
    return per_edge[["pose_label", "prot_resi", "resnr_label", "scoretype", "weighted_energy"]].copy()


def plot_positive_residue_decomposition(protein: str):
    totals = residue_contribution_table(protein)
    if totals.empty:
        print(f"{protein}: no mapped Rosetta contacts")
        return
    decomposed = scoretype_decomposition_table(protein)
    for pose_label in sorted(totals["pose_label"].unique()):
        bad_resi = totals[(totals["pose_label"] == pose_label) & (totals["twobody_total"] > 0)]["prot_resi"]
        if bad_resi.empty:
            continue
        sub = decomposed[(decomposed["pose_label"] == pose_label) & (decomposed["prot_resi"].isin(bad_resi))]
        fig = px.bar(
            sub.sort_values("prot_resi"), x="resnr_label", y="weighted_energy", color="scoretype",
            color_discrete_map=SCORETYPE_COLORS, category_orders=dict(scoretype=SCORETYPE_ORDER),
            labels=dict(weighted_energy="weighted energy (REU)", resnr_label="protein residue number (pocket position)"),
            title=f"{protein} -- {pose_label} -- unfavorable residue decomposition ({len(bad_resi)} residue(s))",
        )
        fig.add_hline(y=0, line_color="black", line_width=1)
        fig.update_xaxes(title="protein residue number (pocket position)", type="category")
        fig.update_yaxes(title="weighted energy (REU)")
        fig.show()


## Importer vs. non-importer energetics

No equivalent in `ga1_pocket_exploration.ipynb` -- that notebook only ever
covers importer proteins (ABCfold never co-folds a non-importer with GA1),
but HADDOCK3 docks GA1 into every redocked complex regardless of role.
One point per redocked complex (`n=15` importer, `n=56` non_importer --
71 total, 1 complex dropped: a real GA1 ring-pucker `RingConformerSet`
failure in PyRosetta, not scored). Per `project_redocking_pipeline_plan`
memory: importer/non_importer barely differ on HADDOCK score, and
importers actually score *worse* on Rosetta `total_score`/`fa_rep` than
non-importers -- worth eyeballing directly rather than trusting only the
pooled means.


In [ ]:
complex_summary = all_contacts.drop_duplicates("complex_id")[
    ["complex_id", "protein", "role", "ca_cluster", "haddock_score", "fa_rep_haddock_pose", "total_score_haddock_pose"]
].copy()


def plot_role_energetics():
    for metric, label in [
        ("haddock_score", "HADDOCK score"),
        ("total_score_haddock_pose", "Rosetta total_score (REU)"),
        ("fa_rep_haddock_pose", "Rosetta fa_rep (REU)"),
    ]:
        fig = px.box(
            complex_summary, x="role", y=metric, points="all", color="role",
            color_discrete_map=ROLE_COLORS, hover_data=["complex_id", "protein", "ca_cluster"],
            title=f"{label} by role -- HADDOCK3-redocked poses",
        )
        fig.update_layout(showlegend=False)
        fig.show()


plot_role_energetics()


## CDD pocket agreement (Rosetta contacts vs. the 35-position putative binding site)

Same precision/recall framework `plip_analysis.py`'s `cdd_agreement()`
uses for PLIP-vs-CDD on ab initio poses, computed here on Rosetta's own
energy-graph contacts from the redocked poses (`rescore_redocked_aggregate.py`'s
`cdd_agreement.csv`) -- **precision** = of the residues Rosetta actually
found interacting with GA1 in this protein's redocked pose(s), what
fraction fall inside the CDD pocket; **recall** = of the 35 CDD positions,
how many are ever actually contacted. Colored by role -- per
`project_redocking_pipeline_plan` memory, these barely separate importer
from non-importer here, likely because HADDOCK3's own restraints
(`define_active_passive.py`'s AIRs) are built from the SAME CDD active-
residue set for both roles.


In [ ]:
cdd_agreement = pd.read_csv(RESCORING_ROOT / "cdd_agreement.csv")


def plot_cdd_agreement():
    sub = cdd_agreement.sort_values(["role", "protein"]).reset_index(drop=True)
    x = list(range(len(sub)))
    fig = go.Figure()
    fig.add_bar(x=[i - 0.2 for i in x], y=sub["precision"], width=0.4, name="precision",
                marker_color="#1f77b4")
    fig.add_bar(x=[i + 0.2 for i in x], y=sub["recall"], width=0.4, name="recall",
                marker_color="#ff7f0e")
    fig.update_xaxes(
        tickmode="array", tickvals=x,
        ticktext=[f"{row.protein} ({row.role})" for row in sub.itertuples()],
        tickangle=90,
    )
    fig.update_yaxes(range=[0, 1.05], title="fraction")
    fig.update_layout(
        title="Rosetta-contact CDD-pocket agreement -- HADDOCK3-redocked poses, importer vs. non_importer",
        height=550, margin=dict(b=180),
    )
    fig.show()


plot_cdd_agreement()

for role in ["importer", "non_importer"]:
    sub = cdd_agreement[cdd_agreement["role"] == role]
    precision = sub["n_in_cdd_pocket"].sum() / sub["n_unique_contacted_residues"].sum()
    recall = sub["n_cdd_positions_contacted"].sum() / sub["n_cdd_positions_total"].sum()
    print(f"{role}: pooled precision={precision:.3f}, pooled recall={recall:.3f}")


## Sequence-LDA importance vs. redocked Rosetta energy

For every CDD position actually contacted in >=1 redocked complex: its
mean Rosetta two-body energy there (across that protein's `ca_cluster`
complexes where contacted) against `NPF_LDA_kernel`'s sequence-derived
importance for that position (`rescore_redocked_aggregate.py`'s
`lda_unfavorable_contacts.csv`). Points above the black line (`unfavorable`,
red) are positions the sequence-only classifier flags as relevant but
where GA1's own physically-docked pose actually clashes rather than
favorably contacts -- interesting regardless of role, but especially so
for high-importance importer positions.


In [ ]:
lda_unfavorable = pd.read_csv(RESCORING_ROOT / "lda_unfavorable_contacts.csv")


def plot_lda_vs_energy():
    fig = px.scatter(
        lda_unfavorable, x="lda_importance", y="mean_twobody_total", color="unfavorable",
        facet_col="role", hover_data=["protein", "position", "n"],
        color_discrete_map={True: "#d62728", False: "#1f77b4"},
        labels=dict(lda_importance="sequence-LDA importance", mean_twobody_total="mean two-body total (REU)"),
        title="Sequence-LDA importance vs. mean Rosetta two-body energy at that CDD position -- redocked poses",
    )
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.show()


plot_lda_vs_energy()

n_unfavorable = int(lda_unfavorable["unfavorable"].sum())
print(f"{n_unfavorable}/{len(lda_unfavorable)} (protein, position) rows flagged unfavorable "
      f"({lda_unfavorable.groupby('role')['unfavorable'].mean().round(3).to_dict()} by role)")


## Does good-pose filtering change any of this?

Every section above scores exactly one model per complex --
`rescore_redocked_batch.py`'s plain best-HADDOCK-score pick. The user
noticed (eyeballing the non-importer pocket-contact box plot further
below) that a real fraction of ALL kept models are off-target/"membrane"
poses, not just the top few -- confirmed in
`redocking/src/pose_pocket_engagement.py` (Stage 8): a GMM(2) on CDD
contact count across every kept model of every complex is sharply
bimodal (component means 0.52 vs 11.43 contacts). Question: does
re-picking each complex's representative model from the "good"
(pocket-engaging) component only -- instead of the plain best-HADDOCK-
score pick -- change the energy profile or CDD-agreement picture above?

**Checked directly, not assumed**: `good_pose_representative.csv` shows
only **2/72 complexes** actually pick a different model once restricted
to good poses (both `non_importer`, both cases where only 1-2 of that
complex's 40 kept models were ever classified "good" at all) -- every
other complex's plain top-1 pick was ALREADY a good pose. Rescored those
2 (and, for a clean apples-to-apples pooled comparison, everything else
too, which reproduces identically) into
`results/rescoring/good_pose_filtered/` via:

```bash
python rescore_redocked_batch.py \
    --out-dir ../results/rescoring/per_complex_good_pose_filtered \
    --representative-csv ../../redocking/results/comparison/good_pose_representative.csv
python rescore_redocked_aggregate.py \
    --per-complex-dir ../results/rescoring/per_complex_good_pose_filtered \
    --out-dir ../results/rescoring/good_pose_filtered
```


In [ ]:
from IPython.display import display

FILTERED_ROOT = RESCORING_ROOT / "good_pose_filtered"

filtered_contacts = pd.read_csv(FILTERED_ROOT / "all_contacts.csv")
filtered_cdd_agreement = pd.read_csv(FILTERED_ROOT / "cdd_agreement.csv")
filtered_complex_summary = filtered_contacts.drop_duplicates("complex_id")[
    ["complex_id", "protein", "role", "haddock_score", "fa_rep_haddock_pose", "total_score_haddock_pose"]
].copy()


def _pooled(agreement: pd.DataFrame, role: str) -> tuple[float, float]:
    sub = agreement[agreement["role"] == role]
    precision = sub["n_in_cdd_pocket"].sum() / sub["n_unique_contacted_residues"].sum()
    recall = sub["n_cdd_positions_contacted"].sum() / sub["n_cdd_positions_total"].sum()
    return precision, recall


comparison_rows = []
for role in ["importer", "non_importer"]:
    orig_p, orig_r = _pooled(cdd_agreement, role)
    filt_p, filt_r = _pooled(filtered_cdd_agreement, role)
    comparison_rows.append(dict(role=role, precision_original=orig_p, precision_good_pose_filtered=filt_p,
                                 recall_original=orig_r, recall_good_pose_filtered=filt_r))
pooled_comparison = pd.DataFrame(comparison_rows)
display(pooled_comparison)

# NOTE: each run independently hits PyRosetta's GA1 ring-pucker
# `RingConformerSet` failure on a DIFFERENT single complex (confirmed by
# hand: the original run failed NPF5.11_Q8RX67__ca0_alphafold3_334df553,
# the good-pose-filtered rerun instead failed
# NPF8.2_Q9LFB8__ca2_rosettafold3_1c4ec380 -- neither is one of the 2
# complexes good-pose filtering actually changes the representative for,
# so this is a separate, apparently RNG/floating-point-boundary-sensitive
# ring-pucker classification quirk, not a filtering effect). The inner
# merge below naturally compares only the complexes present in BOTH runs.
merged_scores = complex_summary.merge(
    filtered_complex_summary, on=["complex_id", "protein", "role"], suffixes=("_original", "_good_pose_filtered"),
)
changed_complexes = merged_scores[merged_scores["haddock_score_original"] != merged_scores["haddock_score_good_pose_filtered"]]
print(f"\ncompared {len(merged_scores)} complexes present in both runs "
      f"({len(complex_summary)} in the original, {len(filtered_complex_summary)} in the filtered rerun -- "
      "1 complex on each side hit the unrelated ring-pucker failure above, not shared)")
print(f"{len(changed_complexes)}/{len(merged_scores)} complexes' representative model actually changed:")
display(changed_complexes[[
    "complex_id", "role",
    "haddock_score_original", "haddock_score_good_pose_filtered",
    "total_score_haddock_pose_original", "total_score_haddock_pose_good_pose_filtered",
    "fa_rep_haddock_pose_original", "fa_rep_haddock_pose_good_pose_filtered",
]])

print("\nConclusion: at the single-best-representative-model level, good-pose filtering leaves the "
      "importer numbers untouched (0 importer complexes affected) and shifts the non_importer pooled "
      "numbers only marginally -- the plain best-HADDOCK-score pick was already a real pocket-engaging "
      "pose in 70/72 complexes. Filtering matters far more at the FULL-ENSEMBLE level (see "
      "results/comparison/pose_pocket_engagement.csv and haddock_ligand_pose_clustering.ipynb) than for "
      "the single representative model this notebook's energy-profile/CDD-agreement sections are built on.")


## Stage 7: HADDOCK3 vs. ABCfold

- **Importer**: ligand heavy-atom RMSD between HADDOCK3's top-ranked model
  and ABCfold's own predicted GA1 pose, after superposing the two
  receptors' C-alpha atoms (`redocking/src/compare_to_abcfold.py`).
  `receptor_ca_superposition_rmsd` (read from each complex's own
  `*_comparison.json`, not in `summary.csv`) is shown alongside as context
  -- a large receptor RMSD would mean the ligand RMSD isn't a
  clean like-for-like comparison.
- **Non-importer**: no ABCfold ligand pose exists to compare against --
  instead, for each of HADDOCK3's top-4 models by score (no clustering,
  see `compare_to_abcfold.py`'s module docstring), how many CDD active-
  pocket residues does GA1 actually contact. Read straight from each
  complex's own `*_comparison.json` (richer than `summary.csv`'s single
  "best model" number) so the across-the-ensemble spread is visible, not
  just one point per complex.


In [ ]:
import json

manifest_lookup = all_contacts.drop_duplicates("complex_id").set_index("complex_id")[["protein", "role", "ca_cluster"]]


def load_importer_rmsd() -> pd.DataFrame:
    rows = []
    for complex_id, info in manifest_lookup[manifest_lookup["role"] == "importer"].iterrows():
        path = COMPARISON_ROOT / f"{complex_id}_comparison.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        rows.append({
            "complex_id": complex_id, "protein": info["protein"], "ca_cluster": info["ca_cluster"],
            "receptor_ca_superposition_rmsd": data["receptor_ca_superposition_rmsd"],
            "ligand_rmsd_vs_abcfold_pose": data["ligand_rmsd_vs_abcfold_pose"],
        })
    return pd.DataFrame(rows).sort_values(["protein", "ca_cluster"])


def plot_importer_rmsd():
    df = load_importer_rmsd()
    fig = go.Figure()
    fig.add_bar(x=df["complex_id"], y=df["ligand_rmsd_vs_abcfold_pose"], name="ligand RMSD vs. ABCfold pose",
                marker_color="#1f77b4")
    fig.add_scatter(x=df["complex_id"], y=df["receptor_ca_superposition_rmsd"], name="receptor C-alpha superposition RMSD",
                     mode="markers", marker=dict(color="black", size=8, symbol="diamond"))
    fig.update_xaxes(tickangle=90, title="complex")
    fig.update_yaxes(title="RMSD (\u00c5)")
    fig.update_layout(title="Importer: HADDOCK3 top model vs. ABCfold's own predicted GA1 pose", height=500)
    fig.show()
    print(df["ligand_rmsd_vs_abcfold_pose"].describe())


plot_importer_rmsd()


In [ ]:
def load_non_importer_pocket_contacts() -> pd.DataFrame:
    rows = []
    for complex_id, info in manifest_lookup[manifest_lookup["role"] == "non_importer"].iterrows():
        path = COMPARISON_ROOT / f"{complex_id}_comparison.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        for model in data["top_models"]:
            rows.append({
                "complex_id": complex_id, "protein": info["protein"], "ca_cluster": info["ca_cluster"],
                "rank": model["rank"], "haddock_score": model["haddock_score"],
                "n_active_residues_contacted": model["n_active_residues_contacted"],
                "n_active_residues_total": model["n_active_residues_total"],
            })
    return pd.DataFrame(rows)


def plot_non_importer_pocket_contacts():
    df = load_non_importer_pocket_contacts()
    fig = px.box(
        df, x="rank", y="n_active_residues_contacted", points="all",
        hover_data=["complex_id", "protein", "haddock_score"],
        title="Non-importer: CDD active-residue contacts per top-4-by-score HADDOCK3 model "
              f"(out of {df['n_active_residues_total'].iloc[0]:.0f} CDD positions)",
        labels=dict(rank="HADDOCK-score rank (1 = best)", n_active_residues_contacted="CDD residues contacted"),
    )
    fig.show()
    print(df.groupby("rank")["n_active_residues_contacted"].describe()[["mean", "min", "max"]])


plot_non_importer_pocket_contacts()


## Interactive: pick one protein


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

_out = widgets.Output()
_dropdown = widgets.Dropdown(options=GA1_PROTEINS, description="protein:")


def _on_change(change):
    with _out:
        clear_output(wait=True)
        protein = change["new"]
        role = complex_summary.loc[complex_summary["protein"] == protein, "role"].iloc[0]
        print(f"{protein}  ({role})")
        plot_residue_contribution(protein)
        plot_positive_residue_decomposition(protein)


_dropdown.observe(_on_change, names="value")
display(_dropdown, _out)
_on_change({"new": GA1_PROTEINS[0]})


## Every protein, one after another

Same per-protein plots as above, looped over all 24 proteins (no
dropdown) -- useful for a top-to-bottom scan or for exporting the whole
notebook to HTML.


In [ ]:
for _protein in GA1_PROTEINS:
    plot_residue_contribution(_protein)
    plot_positive_residue_decomposition(_protein)
